<a href="https://colab.research.google.com/github/DhimanTarafdar/restoration-and-enhancement-ECG-Images/blob/main/ECG_image_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download sample ECG images (Dataset A + Dataset B) from GenECG dataset on Hugging Face.

- Dataset A -> clean ECG images (no imperfections) -> used as ground truth

- Dataset B -> same ECGs but with imperfections (looks like a photographed ECG) -> used as input

We only download a small random sample (not all 21,799 images) because that is enough

In [3]:
! pip install huggingface_hub

In [4]:
import os
import random
from huggingface_hub import HfApi, hf_hub_download

# ---------------------------
# Basic settings (change here if needed)
# ---------------------------
REPO_ID = "edcci/GenECG"
FOLDER_A = "Dataset_A_ECGs_without_imperfections"
FOLDER_B = "Dataset_B_ECGs_with_imperfections"

NUM_SAMPLES = 100          # how many image pairs we want
SAVE_DIR_A = "genecg_sample/dataset_A_clean"
SAVE_DIR_B = "genecg_sample/dataset_B_imperfect"

random.seed(42)  # fixed seed so the same random images get picked every time we run this

In [5]:

# ---------------------------
# Step 1: List all files in the dataset repo (just file names, not actual download yet)
# ---------------------------
api = HfApi()
print("Fetching file list from Hugging Face... this may take a few seconds.")
all_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")

# keep only files that belong to Dataset A and Dataset B folders
files_a = [f for f in all_files if f.startswith(FOLDER_A)]
files_b = [f for f in all_files if f.startswith(FOLDER_B)]

print(f"Found {len(files_a)} images in Dataset A")
print(f"Found {len(files_b)} images in Dataset B")

Fetching file list from Hugging Face... this may take a few seconds.
Found 21799 images in Dataset A
Found 21799 images in Dataset B


In [6]:

# ---------------------------
# Step 2: Match Dataset A and Dataset B files using their ECG id (file name)
# Example: Dataset_A_.../02000_hr_1R.png  and  Dataset_B_.../02000_hr_1R.png
# ---------------------------
names_a = {os.path.basename(f) for f in files_a}
names_b = {os.path.basename(f) for f in files_b}

common_names = list(names_a & names_b)  # file names that exist in BOTH datasets
print(f"Total matching ECG image pairs available: {len(common_names)}")

Total matching ECG image pairs available: 21799


In [7]:
# ---------------------------
# Step 3: Randomly pick N pairs (random = good mix of mild/severe distortion)
# ---------------------------
selected_names = random.sample(common_names, NUM_SAMPLES)

# make folders to save the downloaded images
os.makedirs(SAVE_DIR_A, exist_ok=True)
os.makedirs(SAVE_DIR_B, exist_ok=True)